<a href="https://colab.research.google.com/github/monicamtzmdz86-ux/set-up-dashboard-mvp/blob/main/set_up_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd

df = pd.read_excel('/content/drive/MyDrive/Caso práctico Service centers (1) (1).xlsx',
                   sheet_name='service_centers_dataset.csv')

df.head()

Mounted at /content/drive


,id_sc,nombre_sc,estado,tipo_sc,m2_operativos,throughput_diario_meta,throughput_real_ult_semana,avance_obra_civil_pct,avance_compras_pct,avance_licencias_pct,...,fecha_apertura_real,dias_retraso,nivel_automatizacion,mix_gm_pct,mix_large_pct,mix_oversize_pct,capex_presupuestado_usd,capex_ejercido_usd,estatus_general,observaciones
0,SC-001,SMX-Norte,CDMX,Mediano,1800,3500,3200,100,95,80,...,NaT,0,semi,72,20,8,850000,780000,En operación piloto,Licencias pendientes de validación final
1,SC-002,SMX-Sur,CDMX,Grande,3200,6000,0,85,60,30,...,NaT,35,semi,70,20,10,1250000,620000,En habilitación,Retraso en entrega de equipamiento por proveedor
2,SC-003,GDL-Zapopan,Jalisco,Mediano,2100,4000,0,100,90,95,...,NaT,18,semi,75,18,7,920000,890000,Pre-apertura,Sorter con falla en instalación — reemplazo en...
3,SC-004,MTY-San Nicolás,Nuevo León,Grande,4000,8000,0,60,40,20,...,NaT,0,full,68,22,10,1800000,540000,En obra civil,Sin retrasos previstos a la fecha
4,SC-005,QRO-Centro,Querétaro,Pequeño,750,1500,0,100,100,100,...,2026-02-05,0,manual,78,15,7,380000,395000,Operativo,Apertura completada — sobre presupuesto por aj...


In [43]:
# Limpieza

# Guardamos el número de filas inicial para comparar
filas_iniciales = len(df)

# Eliminar filas completamente vacías
df = df.dropna(how='all')

# Eliminar filas duplicadas por id_sc (mismo centro repetido por error)
df = df.drop_duplicates(subset=['id_sc'])

# Limpiar espacios en blanco en columnas de texto
columnas_texto = ['id_sc', 'nombre_sc', 'estado', 'tipo_sc', 'estatus_general']
for col in columnas_texto:
    df[col] = df[col].str.strip()

# Validar que los porcentajes de avance estén entre 0 y 100
columnas_pct = ['avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct']
for col in columnas_pct:
    df[col] = df[col].clip(lower=0, upper=100)

# Reporte de limpieza
filas_finales = len(df)
print(" Limpieza de datos completada")
print(f"Filas iniciales: {filas_iniciales}")
print(f"Filas finales: {filas_finales}")
print(f"Filas eliminadas (vacías/duplicadas): {filas_iniciales - filas_finales}")
print()
print("Valores nulos por columna después de limpieza:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("Nota: Los nulos en 'fecha_apertura_real' son legítimos (SCs aún no abiertos)")

 Limpieza de datos completada
Filas iniciales: 12
Filas finales: 12
Filas eliminadas (vacías/duplicadas): 0

Valores nulos por columna después de limpieza:
fecha_apertura_real    11
dtype: int64

Nota: Los nulos en 'fecha_apertura_real' son legítimos (SCs aún no abiertos)


In [44]:

# tamaño
print("Tamaño del dataset:")
print(f"  - Filas: {df.shape[0]}")
print(f"  - Columnas: {df.shape[1]}")
print()

# tipo de datos
print("Tipos de datos por columna:")
print(df.dtypes)
print()

# valores faltantes
print("Valores faltantes por columna:")
print(df.isnull().sum())

Tamaño del dataset:
  - Filas: 12
  - Columnas: 21

Tipos de datos por columna:
id_sc                                 object
nombre_sc                             object
estado                                object
tipo_sc                               object
m2_operativos                          int64
throughput_diario_meta                 int64
throughput_real_ult_semana             int64
avance_obra_civil_pct                  int64
avance_compras_pct                     int64
avance_licencias_pct                   int64
fecha_estimada_apertura       datetime64[ns]
fecha_apertura_real           datetime64[ns]
dias_retraso                           int64
nivel_automatizacion                  object
mix_gm_pct                             int64
mix_large_pct                          int64
mix_oversize_pct                       int64
capex_presupuestado_usd                int64
capex_ejercido_usd                     int64
estatus_general                       object
observaciones       

In [45]:


# Avance total promedio de las 3 categorías

df['avance_total_pct'] = df[
['avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct']
].mean(axis=1).round(1)

# Validar días de retraso
hoy = pd.to_datetime('today').normalize()
df['dias_retraso_calculado'] = (hoy - df['fecha_estimada_apertura']).dt.days

df['dias_retraso_calculado'] = df['dias_retraso_calculado'].clip(lower=0)

# semáforo
def clasificar_retraso(dias):
    if dias < 7:
        return 'Verde (< 7 días)'
    elif dias <= 14:
        return 'Amarillo (7-14 días)'
    else:
        return 'Rojo (> 14 días)'

df['categoria_retraso'] = df['dias_retraso'].apply(clasificar_retraso)

# Desviación de presupuesto (capex)
df['desviacion_capex_pct'] = (
    (df['capex_ejercido_usd'] - df['capex_presupuestado_usd'])
    / df['capex_presupuestado_usd'] * 100
).round(1)

# throughput operativo
df['cumplimiento_throughput_pct'] = (
    df['throughput_real_ult_semana'] / df['throughput_diario_meta'] * 100
).round(1)

#
df[['id_sc', 'estado', 'tipo_sc', 'avance_total_pct',
    'dias_retraso', 'categoria_retraso', 'estatus_general']]

,id_sc,estado,tipo_sc,avance_total_pct,dias_retraso,categoria_retraso,estatus_general
0,SC-001,CDMX,Mediano,91.7,0,Verde (< 7 días),En operación piloto
1,SC-002,CDMX,Grande,58.3,35,Rojo (> 14 días),En habilitación
2,SC-003,Jalisco,Mediano,95.0,18,Rojo (> 14 días),Pre-apertura
3,SC-004,Nuevo León,Grande,40.0,0,Verde (< 7 días),En obra civil
4,SC-005,Querétaro,Pequeño,100.0,0,Verde (< 7 días),Operativo
5,SC-006,Puebla,Mediano,56.7,14,Amarillo (7-14 días),En habilitación
6,SC-007,San Luis Potosí,Pequeño,83.3,8,Amarillo (7-14 días),Pre-apertura
7,SC-008,Baja California,Grande,28.3,0,Verde (< 7 días),En obra civil
8,SC-009,CDMX,Especializado,75.0,21,Rojo (> 14 días),En habilitación
9,SC-010,Veracruz,Pequeño,95.0,5,Verde (< 7 días),Pre-apertura


In [46]:

columnas_orden = [
    'id_sc', 'nombre_sc', 'estado', 'tipo_sc',
    'avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct',
    'avance_total_pct',
    'fecha_estimada_apertura', 'fecha_apertura_real',
    'dias_retraso', 'categoria_retraso',
    'estatus_general',
    'm2_operativos', 'throughput_diario_meta', 'throughput_real_ult_semana',
    'cumplimiento_throughput_pct',
    'capex_presupuestado_usd', 'capex_ejercido_usd', 'desviacion_capex_pct',
    'nivel_automatizacion',
    'mix_gm_pct', 'mix_large_pct', 'mix_oversize_pct',
    'observaciones'
]

df_final = df[columnas_orden]

df_final.to_excel('service_centers_consolidado.xlsx', index=False)

print("✅ Archivo Excel generado exitosamente")
print(f"   Filas: {len(df_final)}")
print(f"   Columnas: {len(df_final.columns)}")

# Vista previa
df_final.head()

✅ Archivo Excel generado exitosamente
   Filas: 12
   Columnas: 25


,id_sc,nombre_sc,estado,tipo_sc,avance_obra_civil_pct,avance_compras_pct,avance_licencias_pct,avance_total_pct,fecha_estimada_apertura,fecha_apertura_real,...,throughput_real_ult_semana,cumplimiento_throughput_pct,capex_presupuestado_usd,capex_ejercido_usd,desviacion_capex_pct,nivel_automatizacion,mix_gm_pct,mix_large_pct,mix_oversize_pct,observaciones
0,SC-001,SMX-Norte,CDMX,Mediano,100,95,80,91.7,2026-02-15,NaT,...,3200,91.4,850000,780000,-8.2,semi,72,20,8,Licencias pendientes de validación final
1,SC-002,SMX-Sur,CDMX,Grande,85,60,30,58.3,2026-04-10,NaT,...,0,0.0,1250000,620000,-50.4,semi,70,20,10,Retraso en entrega de equipamiento por proveedor
2,SC-003,GDL-Zapopan,Jalisco,Mediano,100,90,95,95.0,2026-03-01,NaT,...,0,0.0,920000,890000,-3.3,semi,75,18,7,Sorter con falla en instalación — reemplazo en...
3,SC-004,MTY-San Nicolás,Nuevo León,Grande,60,40,20,40.0,2026-06-20,NaT,...,0,0.0,1800000,540000,-70.0,full,68,22,10,Sin retrasos previstos a la fecha
4,SC-005,QRO-Centro,Querétaro,Pequeño,100,100,100,100.0,2026-01-30,2026-02-05,...,0,0.0,380000,395000,3.9,manual,78,15,7,Apertura completada — sobre presupuesto por aj...


In [47]:
#  librería de Anthropic

# Instalar el SDK oficial de Anthropic (Claude)
!pip install anthropic --quiet

print("✅ Librería anthropic instalada correctamente")

✅ Librería anthropic instalada correctamente


In [49]:
# IA + EXPORT A GOOGLE SHEETS


# Instalar librería
!pip install anthropic gspread gspread_dataframe --quiet

# Imports
import anthropic
import json
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import userdata, auth
from google.auth import default

# AUTENTICACIÓN

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Abrir archivo de Google Sheets
spreadsheet = gc.open("Set-up Dash")
worksheet = spreadsheet.sheet1

# ESCRIBIR DATAFRAME

print("📤 Enviando DataFrame a Google Sheets...")

worksheet.clear()  # limpia antes de escribir
set_with_dataframe(worksheet, df_final)

print(f"✅ Datos cargados al sheet correctamente ({len(df_final)} filas)")

# CONFIGURAR CLAUDE

api_key = userdata.get('antropic_api_key')
client = anthropic.Anthropic(api_key=api_key)

# Columnas para IA
columnas_para_ia = [
    'id_sc', 'nombre_sc', 'estado', 'tipo_sc',
    'avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct',
    'avance_total_pct', 'dias_retraso', 'categoria_retraso',
    'estatus_general', 'desviacion_capex_pct', 'observaciones'
]

df_para_ia = df_final[columnas_para_ia].copy()
df_json = df_para_ia.to_json(orient='records', force_ascii=False)



def generar_resumen_ejecutivo(datos_json: str) -> str:

    prompt = f"""Eres un analista senior de operaciones logísticas especializado
en apertura de Service Centers. Analiza el siguiente reporte semanal de aperturas
y genera un análisis ejecutivo estructurado.

DATOS DE LOS SERVICE CENTERS:
{datos_json}

INSTRUCCIONES:

Genera tu respuesta en este formato exacto en español:

## RESUMEN EJECUTIVO
Un párrafo de máximo 80 palabras con la situación general de las aperturas.

## TOP 3 RIESGOS OPERATIVOS DE LA SEMANA
- Riesgo
- SCs afectados
- Impacto

## ACCIONES RECOMENDADAS
- Qué hacer
- Quién
- Plazo
"""

    mensaje = client.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=1500,
        messages=[{'role': 'user', 'content': prompt}]
    )

    return mensaje.content[0].text

# EJECUTAR IA

print("\n Enviando datos a Claude...\n")
print("=" * 70)

resumen = generar_resumen_ejecutivo(df_json)
print(resumen)

print("=" * 70)
print("\n Proceso completo: datos + análisis")


📤 Enviando DataFrame a Google Sheets...
✅ Datos cargados al sheet correctamente (12 filas)

 Enviando datos a Claude...

# ANÁLISIS EJECUTIVO SEMANAL - APERTURA DE SERVICE CENTERS

## RESUMEN EJECUTIVO

De 12 Service Centers en portafolio, 1 está operativo y 1 en operación piloto. Se identifican 3 SCs en estado crítico (rojo) con retrasos superiores a 14 días, acumulando 74 días de atraso combinado. El 50% del portafolio mantiene avances inferiores al 60%. Destaca SC-002 con 35 días de retraso y solo 30% en licencias. Positivamente, 7 SCs están en verde sin retrasos. La desviación CAPEX promedio es favorable (-36.6%), aunque SC-005 excedió presupuesto (+3.9%).

## TOP 3 RIESGOS OPERATIVOS DE LA SEMANA

**1. RETRASOS CRÍTICOS EN CADENA DE SUMINISTRO**
- **SCs afectados:** SC-002 (SMX-Sur), SC-003 (GDL-Zapopan), SC-007 (SLP-Industrial)
- **Impacto:** Retrasos de 18-35 días por fallas en proveedores de equipamiento crítico (sorter, mesas clasificación). SC-002 con solo 60% en compras repr

In [50]:
from datetime import datetime

# Guardar el análisis en un archivo de texto
fecha_hoy = datetime.now().strftime('%Y-%m-%d')
Resumen = f'resumen_ejecutivo_{fecha_hoy}.md'

with open(Resumen, 'w', encoding='utf-8') as f:
    f.write(f"# Reporte Ejecutivo de Aperturas - Service Centers\n")
    f.write(f"*Fecha de generación:* {fecha_hoy}\n")
    f.write(f"*Generado por:* Módulo de IA (Claude API)\n\n")
    f.write("---\n\n")
    f.write(resumen)

print(f"✅ Resumen guardado en: {Resumen}")

✅ Resumen guardado en: resumen_ejecutivo_2026-05-25.md


In [51]:
#Versión compacta del resumen para el tablero

prompt_compacto = f"""Eres analista de operaciones. Con estos datos de 12 Service Centers,
escribe un resumen MUY breve para un tablero ejecutivo.

DATOS:
{df_json}

FORMATO EXACTO (sin títulos, sin markdown, máximo 6 líneas):
- Una frase con el panorama (avance promedio y SCs en riesgo crítico).
- Los 3 riesgos principales, uno por línea, empezando con "• ", máximo 12 palabras cada uno.
- Una frase final con la acción prioritaria de la semana.

Lenguaje directo, sin tecnicismos. No uses negritas ni símbolos ##."""

msg = client.messages.create(
    model='claude-sonnet-4-5',
    max_tokens=400,
    messages=[{'role': 'user', 'content': prompt_compacto}]
)

resumen_corto = msg.content[0].text
print(resumen_corto)

El avance promedio es 65.3% con 3 Service Centers en retraso crítico (>14 días) que requieren intervención inmediata.

• SC-002 SMX-Sur: 35 días de retraso por falta de equipamiento del proveedor.
• SC-009 MER-Centro: 21 días atrasado, necesita expansión para zona de voluminosos.
• SC-006 PUE-Angelópolis: demora en permisos municipales con riesgo de +30 días adicionales.

Esta semana es prioritario escalar con proveedores de SC-002 y desbloquear permisos de SC-006 para evitar mayor impacto en cronograma y presupuesto.


In [52]:
#  Resumen_IA
from datetime import datetime
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

# Abrir tu hoja por URL

URL_HOJA = 'https://docs.google.com/spreadsheets/d/1NmYU54z4G4kO5ivkbI-UWAOC9PLqEkjRcS48PQH65NQ/edit?gid=834092528#gid=834092528'
sh = gc.open_by_url(URL_HOJA)

fecha = datetime.now().strftime('%Y-%m-%d')

try:
    ws = sh.worksheet('Resumen_IA')
    ws.clear()
except gspread.exceptions.WorksheetNotFound:
    ws = sh.add_worksheet(title='Resumen_IA', rows=50, cols=1)

ws.update_acell('A1', f"Resumen IA · {fecha}\n\n{resumen_corto}")
print('Resumen compacto escrito en la hoja')

Resumen compacto escrito en la hoja
